In [ ]:
# =============================================================================
# ai_service.py - AI 분석 서비스
# =============================================================================
# 담당: AI
# 설명: OCR 추출 텍스트를 LLM으로 분석하여 구조화
#       - LLM API 호출 (OpenAI / Anthropic 등)
#       - 비정형 텍스트 → 정형 데이터 변환
#       - 카테고리 자동 분류
# =============================================================================

In [1]:
!pip install google-generativeai

Defaulting to user installation because normal site-packages is not writeable


In [2]:
# 분석 데이터 로드
import json

with open("../dummy data/dummy_receipts_full_format.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(data)

[{'id': 1, 'user_id': 1, 'category_id': 4, 'payment_method_id': 2, 'date': '2024-01-16', 'total_amount': 28173, 'store_name': '배달의민족', 'image_path': 'data/receipts/img001.jpg', 'details': [{'name': '라떼', 'quantity': 1, 'price': 28173}], 'created_at': '2024-01-16T00:00:00'}, {'id': 2, 'user_id': 1, 'category_id': 6, 'payment_method_id': 1, 'date': '2024-01-17', 'total_amount': 10036, 'store_name': '스타벅스', 'image_path': 'data/receipts/img002.jpg', 'details': [{'name': '전기장판', 'quantity': 2, 'price': 5018}], 'created_at': '2024-01-17T00:00:00'}, {'id': 3, 'user_id': 1, 'category_id': 4, 'payment_method_id': 1, 'date': '2024-01-18', 'total_amount': 37925, 'store_name': '무신사', 'image_path': 'data/receipts/img003.jpg', 'details': [{'name': '치킨', 'quantity': 1, 'price': 37925}], 'created_at': '2024-01-18T00:00:00'}, {'id': 4, 'user_id': 1, 'category_id': 4, 'payment_method_id': 1, 'date': '2024-01-19', 'total_amount': 15644, 'store_name': '이마트', 'image_path': 'data/receipts/img004.jpg', 'deta

In [6]:
# Gemini 연결
import google.generativeai as genai
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

genai.configure(api_key=api_key)

model = genai.GenerativeModel("gemini-2.5-flash")

In [7]:
# 프롬프트 생성

prompt = f"""
다음은 사용자의 소비 데이터이다.

{json.dumps(data, ensure_ascii=False, indent=2)}

다음 내용을 분석해라.

1. 전체 소비 패턴
2. 과소비 카테고리
3. 월별 소비 변화
4. 절약을 위한 조언
"""

In [8]:
# google Geminai 분석 실행

response = model.generate_content(prompt)

print(response.text)

사용자님의 소비 데이터를 분석하여 다음 내용을 정리했습니다.

---

### 소비 데이터 분석 (2024-01-16 ~ 2024-02-04)

**총 소비 금액:** 851,920원
**총 거래 건수:** 20건
**분석 기간:** 약 20일 (1월 16일 ~ 2월 4일)

---

### 1. 전체 소비 패턴

*   **총 소비액 및 기간:** 약 20일 동안 총 851,920원을 소비했으며, 이는 하루 평균 약 42,596원의 지출을 의미합니다. (단, 2월 데이터는 4일치만 포함되어 전체 월간 평균과는 다를 수 있습니다.)
*   **주요 소비처:** '이마트'에서 6건으로 가장 많은 거래가 발생했으며, '배달의민족'과 '무신사'에서 각각 4건의 거래가 있었습니다. '김밥천국'에서도 3건의 거래가 나타났습니다.
    *   이마트: 6건
    *   배달의민족: 4건
    *   무신사: 4건
    *   김밥천국: 3건
    *   스타벅스: 1건
    *   쿠팡: 1건
*   **주요 결제 수단:** `payment_method_id` 2번이 13건으로 가장 많이 사용되었고, `payment_method_id` 1번은 7건 사용되었습니다.
*   **주요 구매 품목 (details.name 기준):**
    *   맨투맨: 6회 (총 14개 구매)
    *   치킨: 4회 (총 5개 구매)
    *   라떼: 3회 (총 4개 구매)
    *   전기장판: 3회 (총 6개 구매)
    *   라면: 2회 (총 4개 구매)
    *   청바지: 2회 (총 4개 구매)
    *   아메리카노: 2회 (총 5개 구매)
    *   *전반적으로 식료품/외식과 의류 구매가 높은 빈도를 보입니다.*

---

### 2. 과소비 카테고리

제공된 데이터에는 `category_id`의 구체적인 명칭이 없어 정확한 카테고리 분류는 어렵지만, 지출 비중이 높은 상위 카테고리를 기준으로 분석할 수 있습니다. `details.name`을 통해